# ViFinQA — build the BGE-M3 row-label index (Step 4)

Encode-only job: **much cheaper than generation** (~10–20 min on one T4) and the
result is reusable by every later codegen run.

**Settings:** Accelerator = GPU T4 (x1 is enough), Internet = On, Add Input → `vifinqa-payload`.

Output: `label_index/{labels.npy,labels.json}` → download and place into
`artifacts/store/label_index/` locally, then rerun `scripts/04_make_kaggle_payload.py`
so the index ships (and gets SHA-256 fingerprinted) with the next payload.

In [ ]:
import glob, pathlib
hits = glob.glob("/kaggle/input/**/payload-manifest.json", recursive=True)
assert hits, "Chua attach dataset vifinqa-payload"
PAYLOAD = str(pathlib.Path(hits[0]).parent)
print("PAYLOAD =", PAYLOAD)

In [ ]:
%%time
!pip install -q -U sentence-transformers
import sentence_transformers, torch
print(sentence_transformers.__version__, torch.cuda.get_device_name(0))

In [ ]:
%%time
import sys
sys.path.insert(0, PAYLOAD + "/code")
from pathlib import Path
from vifinqa.extraction.build_store import Store
from vifinqa.retrieval.dense import LabelEncoder, collect_labels

store = Store(Path(PAYLOAD) / "store", cache_size=2)
labels = collect_labels(store)
print("distinct labels:", len(labels))
enc = LabelEncoder("BAAI/bge-m3", cache_dir=None, device="cuda", batch_size=128)
enc.build_cache(labels, Path("/kaggle/working/label_index"))

In [ ]:
# quick sanity: does the index rank the right row above lexical noise?
probe = ["tra truoc cho nguoi ban", "lai tien gui", "du phong rui ro cho vay khach hang"]
cands = [l for l in labels if "Trả trước" in l or "Lãi tiền gửi" in l or "Dự phòng rủi ro" in l][:40]
sims = enc.similarity(probe, cands)
for l, s in sorted(sims.items(), key=lambda x: -x[1])[:10]:
    print(f"{s:.3f}  {l[:80]}")

Tải `label_index/` từ panel Output → đặt vào `artifacts/store/label_index/` ở local →
`python scripts/04_make_kaggle_payload.py` → upload payload mới.
Sau đó chạy codegen với `--use-dense`.